# Challenge 4 — Answer / Verify / Refine Agent Workflow

An agent that answers cloud-architecture questions but **verifies and refines**
the answer before returning it.

Built with the modern ADK graph API (`google.adk.Workflow`) rather than the
deprecated `SequentialAgent` / `LoopAgent` templates. In ADK 2.4.0 those emit:

> `DeprecationWarning: SequentialAgent is deprecated in favor of Workflow`

## Topology

```
START -> intake_agent -> research_agent -> critic_agent -> route_review
         route_review --REFINE--> refine_agent --> critic_agent   (loop back)
         route_review --DONE----> publish_answer                  (terminal)
```

`route_review` is the gate. A graph cycle is legal because the loop contains a
routed edge — ADK only rejects cycles built entirely from unrouted edges.

## The state chain

State is what lets each agent build on the previous one. Written with
`output_key`, read back with `{key}` templating inside the next agent's
instruction.

| Key              | Written by                       | Read by                          |
|------------------|----------------------------------|----------------------------------|
| `research_brief` | `intake_agent`                   | `critic_agent`, `refine_agent`   |
| `draft_answer`   | `research_agent`, `refine_agent` | `refine_agent`, `publish_answer` |
| `critique`       | `critic_agent`                   | forwarded as `route_review` input |
| `revisions`      | `route_review`                   | `route_review` (loop cap)        |

Each node is a `single_turn` agent, so ADK forces `include_contents='none'` —
a node sees only its own input, never the whole transcript. That keeps the
critic focused on the draft instead of re-reading the conversation.

## How to run

1. **Run cell 1** — sets the Vertex AI backend and asserts it took effect. It
   prompts for a GCP project id if `GOOGLE_CLOUD_PROJECT` is not already set.
   No API key is used; auth is ambient ADC.
2. **Run cell 2** — defines the four agents (intake, research, critic, refine).
3. **Run cell 3** — defines the routing gate and compiles the graph. It prints
   the node list and the terminal node, which is a cheap wiring check.
4. **Run cell 4** — builds the `Runner` and the `ask_answer_desk` helper.
5. **Run cell 5** — the live demo. Prints a per-node event trace so you can
   watch each sub-agent contribute in order. Takes 30-60s and several LLM calls.
6. **Run cell 6** — prints `session.state` to show the state chain that the
   agents actually built.
7. **Run cell 7** — the tests. `TestLiveRun` asserts against the cell 5 result,
   so cells 5 and 6 must have run first.

Restart-and-Run-All works top to bottom.

## 1. Configuration and Vertex AI authentication

In [1]:
import os

from google import genai
from google.genai import types

from google.adk import Agent, Event, Workflow
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search

GOOGLE_CLOUD_PROJECT = (
    os.environ.get("GOOGLE_CLOUD_PROJECT") or input("GCP project id: ").strip()
)

# Vertex AI via ambient ADC. No Gemini API key is involved -- a stray
# GOOGLE_API_KEY (e.g. a Maps key) would shadow ADC and cause a confusing
# 403 API_KEY_SERVICE_BLOCKED, so both key vars are cleared.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"   # must be "1"; the default is "0"
os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ.pop("GOOGLE_API_KEY", None)
os.environ.pop("GEMINI_API_KEY", None)

_probe = genai.Client()
assert _probe.vertexai, "Not on the Vertex AI backend -- check GOOGLE_GENAI_USE_VERTEXAI."
print("Vertex AI backend OK | project:", GOOGLE_CLOUD_PROJECT)

MODEL = "gemini-2.5-flash"

# Cap the refine loop. Each pass costs a critic call plus a refine call, so
# keep this small to stay clear of per-minute token quota.
MAX_REVISIONS = 2

Vertex AI backend OK | project: qwiklabs-gcp-03-aa9fafb9374b


## 2. The four agents

Design notes, both learned from reading the reference solutions:

- The greeter is **load-bearing**. It does not just say hello; it converts a
  vague ask into a structured brief with constraints and sub-questions. A
  decorative greeter burns an LLM call and produces nothing downstream.
- The critic gets an **explicit rubric** and must emit a machine-readable
  `VERDICT:` line. A critic told merely to "suggest improvements" returns
  praise, the refiner changes nothing, and the whole loop becomes invisible.

In [2]:
intake_agent = Agent(
    name="intake_agent",
    model=MODEL,
    mode="single_turn",
    description="Greets the user and turns a vague cloud question into a research brief.",
    instruction="""You are the intake analyst for a cloud architecture review desk.

The user's message is your input. Do two things, in this order:

1. Greet the user in one short sentence that names the decision they face.
2. Emit a research brief with exactly these headings:
   ASK: one sentence restating the decision to be made.
   CONSTRAINTS: bullets for every constraint the user stated (team size, budget,
     compliance, existing stack, timeline). Write "none stated" if absent.
   SUB-QUESTIONS: 3-4 bullets naming what must be looked up to answer
     responsibly (pricing models, operational burden, lock-in, scaling limits).
   AMBIGUITIES: bullets naming what the user left unspecified that would change
     the recommendation.

Do not answer the question. Do not name a technology you would pick.
Keep the brief under 200 words.""",
    output_key="research_brief",
)

research_agent = Agent(
    name="research_agent",
    model=MODEL,
    mode="single_turn",
    description="Researches the brief with Google Search and drafts a recommendation.",
    instruction="""You are a cloud solutions architect. Your input is a research brief.

Use the google_search tool to look up current, concrete information for the
SUB-QUESTIONS in the brief: pricing models, operational burden, scaling limits,
managed-service alternatives. Do not rely on memory for anything numeric or
version-specific.

Then write a draft with these headings:
  RECOMMENDATION: the architecture you would choose, in one or two sentences.
  WHY: 3-4 bullets, each tied to a constraint from the brief.
  TRADEOFFS: 2-3 bullets naming what this choice costs or gives up.
  WHAT WOULD CHANGE MY MIND: 2 bullets.

Rules: respect every CONSTRAINT in the brief. Attribute any figure or product
claim to what you actually found in search. Never invent prices or SLAs.
Under 400 words.""",
    tools=[google_search],
    output_key="draft_answer",
)

critic_agent = Agent(
    name="critic_agent",
    model=MODEL,
    mode="single_turn",
    description="Audits a draft against a fixed rubric and issues a verdict.",
    instruction="""You are a staff engineer reviewing a peer's draft recommendation.
Your input is the draft. The brief it was written against is:

{research_brief}

Audit the draft against this rubric. Be specific and quote the offending phrase.
  1. UNSUPPORTED CLAIMS - assertions with no basis, especially numbers and SLAs.
  2. IGNORED CONSTRAINTS - anything in the brief's CONSTRAINTS the draft walked past.
  3. MISSING COUNTERPOINTS - the strongest argument against the recommendation,
     if the draft failed to make it.
  4. VAGUENESS - advice too generic to act on, such as "consider scalability".
  5. STRUCTURE - missing or padded sections.

Output exactly:
  FINDINGS: numbered list. Each entry names the rubric category, quotes the
    problem, and states the concrete fix. Write "none" for a clean category.
  VERDICT: PASS   (only if nothing left would change the recommendation or
                   mislead the reader)
  VERDICT: REVISE (if any finding must be fixed)

Emit exactly one VERDICT line. Do not rewrite the draft. Under 300 words.""",
    output_key="critique",
)

refine_agent = Agent(
    name="refine_agent",
    model=MODEL,
    mode="single_turn",
    description="Rewrites the draft so every critique finding is resolved.",
    instruction="""You are the original author, revising after review.
Your input is the reviewer's findings.

The brief:
{research_brief}

The draft you are revising:
{draft_answer}

Rewrite the draft so every finding is resolved. Keep the same headings
(RECOMMENDATION / WHY / TRADEOFFS / WHAT WOULD CHANGE MY MIND).

Rules:
  - Fix each finding at its root. Do not bolt on a caveat paragraph.
  - If a claim is called unsupported, either ground it or remove it. Never keep
    a number you cannot support.
  - If a constraint was ignored, make it visibly drive the recommendation.
  - Do not thank the reviewer or narrate your edits. Output only the revised
    recommendation. Under 400 words.""",
    output_key="draft_answer",
)

print("Agents ready:", [a.name for a in (intake_agent, research_agent, critic_agent, refine_agent)])

Agents ready: ['intake_agent', 'research_agent', 'critic_agent', 'refine_agent']


## 3. The routing gate and the graph

`route_review` is a plain function, so ADK wraps it in a `FunctionNode`. Two
things worth knowing about that wrapper:

- Parameters other than `node_input` are bound from **session state**, which is
  how `revisions` reads back the counter it wrote on the previous pass.
- Returning `Event(route=...)` drives which edge fires; `Event(output=...)`
  is the payload handed to the next node; `Event(state=...)` becomes a
  `state_delta` that ADK commits before the next node starts.

In [3]:
def route_review(node_input: str, revisions: int = 0):
    """Gate the review loop: send the draft back to be refined, or release it.

    `revisions` is bound from session state by the FunctionNode, defaulting to
    0 on the first pass.
    """
    critique = node_input or ""
    passed = "VERDICT: PASS" in critique.upper()
    exhausted = revisions >= MAX_REVISIONS

    if passed or exhausted:
        return Event(
            route="DONE",
            output=critique,
            state={
                "revisions": revisions,
                "review_outcome": "PASS" if passed else "REVISION_LIMIT",
            },
        )
    return Event(
        route="REFINE",
        output=critique,
        state={"revisions": revisions + 1, "review_outcome": "REVISE"},
    )


def publish_answer(draft_answer: str = "", review_outcome: str = "", revisions: int = 0):
    """Terminal node: hand the reviewed recommendation back to the user."""
    banner = f"[reviewed: {revisions} refinement pass(es), outcome: {review_outcome}]"
    return Event(message=f"{draft_answer}\n\n{banner}", output=draft_answer)


answer_desk = Workflow(
    name="answer_desk",
    description="Answers cloud-architecture questions, verifying and refining before returning.",
    edges=[
        ("START", intake_agent, research_agent, critic_agent, route_review),
        (route_review, {"REFINE": refine_agent, "DONE": publish_answer}),
        (refine_agent, critic_agent),   # back-edge closes the review loop
    ],
)

print("Nodes:   ", [n.name for n in answer_desk.graph.nodes])
print("Terminal:", answer_desk.graph._terminal_node_names)

Nodes:    ['__START__', 'intake_agent', 'research_agent', 'critic_agent', 'route_review', 'refine_agent', 'publish_answer']
Terminal: {'publish_answer'}


## 4. Runner and event trace

A `Workflow` is the root here. It cannot be nested under a root `LlmAgent` —
ADK 2.4.0 warns *"Workflow cannot yet be used as an LlmAgent sub-agent"* — so
it is passed to `Runner(node=...)`.

`ask_answer_desk` prints `event.node_name` for every text-bearing event, which
is the requirement to "output events demonstrating the use of the sub agents".

In [4]:
APP_NAME = "challenge4_answer_desk"
USER_ID = "workshop_user"

session_service = InMemorySessionService()
runner = Runner(node=answer_desk, app_name=APP_NAME, session_service=session_service)


async def ask_answer_desk(question: str, show_trace: bool = True) -> dict:
    """Run the workflow once, printing each node's contribution as it happens."""
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

    final_text, nodes_seen = "(no response)", []
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=question)]),
    ):
        text = ""
        if event.content and event.content.parts:
            text = "".join(
                p.text for p in event.content.parts if p.text and not p.thought
            )
        if not text.strip():
            continue

        node = event.node_name or event.author
        nodes_seen.append(node)
        if show_trace:
            print(f"--- [{node}] ".ljust(74, "-"))
            print(text.strip())
            print()
        final_text = text.strip()

    return {"session_id": session.id, "final": final_text, "nodes": nodes_seen}


print("Runner ready for workflow:", runner.agent.name)

Runner ready for workflow: answer_desk


## 5. Live demo

A deliberate choice of question: a genuine tradeoff with a stated constraint
(three people, nobody has run Kubernetes) that a first draft will plausibly
overclaim on or ignore. Trivia questions produce praise-only critiques and an
invisible refine step.

In [5]:
QUESTION = (
    "We're a 3-person team shipping a B2B SaaS app to about 40 enterprise "
    "customers. We're on a single VM today and it's creaking. Should we move to "
    "Kubernetes on GKE, or to a managed PaaS like Cloud Run? Budget is tight and "
    "nobody here has run Kubernetes in production."
)

result = await ask_answer_desk(QUESTION)

print("=" * 74)
print("Nodes that ran, in order:")
for i, node in enumerate(result["nodes"], 1):
    print(f"  {i}. {node}")

--- [intake_agent] -------------------------------------------------------
Hello! Let's get started on analyzing your decision about moving to Kubernetes on GKE or a managed PaaS like Cloud Run.

### Research Brief

**ASK:** Should the team migrate their B2B SaaS application from a single VM to Kubernetes on GKE or a managed PaaS like Cloud Run?

**CONSTRAINTS:**
*   **Team size:** 3-person team
*   **Budget:** Budget is tight
*   **Compliance:** none stated
*   **Existing stack:** Single VM today
*   **Timeline:** none stated
*   **Other:** No prior production Kubernetes experience within the team

**SUB-QUESTIONS:**
*   What are the total cost of ownership differences between GKE and Cloud Run for a 3-person team and 40 enterprise customers?
*   What is the operational overhead and required team skillset for each option?
*   How do GKE and Cloud Run support common B2B SaaS requirements like specific networking or data handling?
*   What are the scaling limitations or benefits of each

## 6. The state chain the agents built

Read straight off the session rather than scraped from the event stream, which
is the reliable way to show state actually persisted.

In [6]:
session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=result["session_id"]
)

print("Session state after the run:\n")
for key, value in session.state.items():
    shown = value if isinstance(value, int) else f"{str(value)[:160]}..."
    print(f"  {key:<16} = {shown}")

Session state after the run:

  research_brief   = Hello! Let's get started on analyzing your decision about moving to Kubernetes on GKE or a managed PaaS like Cloud Run.

### Research Brief

**ASK:** Should the...
  draft_answer     = RECOMMENDATION:
Given the team's small size (3 persons), tight budget, and lack of prior Kubernetes experience, I recommend migrating the B2B SaaS application t...
  critique         = FINDINGS:
1.  **UNSUPPORTED CLAIMS** - "This direct alignment of costs with actual usage is crucial for a tight budget, avoiding charges for unused resources, a...
  revisions        = 2
  review_outcome   = REVISION_LIMIT...


## 7. Tests

Structural tests prove the answer/verify/refine graph is wired correctly and
run without spending an LLM call. The gate tests drive `route_review` directly
through all three of its branches. `TestLiveRun` asserts against the cell 5
demo, so cells 5 and 6 must have run first.

In [7]:
import unittest

from google.adk.workflow import START


class TestWorkflowWiring(unittest.TestCase):
    """The graph is wired to answer, then verify, then refine."""

    def setUp(self):
        self.edges = {
            (e.from_node.name, e.to_node.name): e.route for e in answer_desk.graph.edges
        }

    def test_all_four_required_agents_are_nodes(self):
        names = {n.name for n in answer_desk.graph.nodes}
        for required in ("intake_agent", "research_agent", "critic_agent", "refine_agent"):
            self.assertIn(required, names)

    def test_answer_then_verify_pipeline(self):
        for edge in (
            (START.name, "intake_agent"),   # the sentinel resolves to '__START__'
            ("intake_agent", "research_agent"),
            ("research_agent", "critic_agent"),
            ("critic_agent", "route_review"),
        ):
            self.assertIn(edge, self.edges)

    def test_refine_loop_closes_with_a_back_edge(self):
        self.assertEqual(self.edges.get(("route_review", "refine_agent")), "REFINE")
        self.assertEqual(self.edges.get(("route_review", "publish_answer")), "DONE")
        self.assertIn(("refine_agent", "critic_agent"), self.edges)

    def test_exactly_one_terminal_node(self):
        self.assertEqual(answer_desk.graph._terminal_node_names, {"publish_answer"})

    def test_state_chain_keys(self):
        self.assertEqual(intake_agent.output_key, "research_brief")
        self.assertEqual(research_agent.output_key, "draft_answer")
        self.assertEqual(critic_agent.output_key, "critique")
        self.assertEqual(refine_agent.output_key, "draft_answer")


class TestReviewGate(unittest.TestCase):
    """route_review decides whether the answer is good enough to release."""

    def test_revise_routes_to_refine_and_increments(self):
        ev = route_review(node_input="FINDINGS: 1. VAGUENESS\nVERDICT: REVISE", revisions=0)
        self.assertEqual(ev.actions.route, "REFINE")
        self.assertEqual(ev.actions.state_delta["revisions"], 1)

    def test_pass_routes_straight_to_done(self):
        ev = route_review(node_input="FINDINGS: none\nVERDICT: PASS", revisions=0)
        self.assertEqual(ev.actions.route, "DONE")
        self.assertEqual(ev.actions.state_delta["review_outcome"], "PASS")

    def test_revision_limit_forces_done(self):
        ev = route_review(node_input="VERDICT: REVISE", revisions=MAX_REVISIONS)
        self.assertEqual(ev.actions.route, "DONE")
        self.assertEqual(ev.actions.state_delta["review_outcome"], "REVISION_LIMIT")


class TestLiveRun(unittest.TestCase):
    """The demo run above actually exercised the sub-agents."""

    def test_answer_verify_refine_nodes_all_ran(self):
        self.assertLessEqual(
            {"intake_agent", "research_agent", "critic_agent"}, set(result["nodes"])
        )

    def test_final_answer_is_a_recommendation(self):
        self.assertIn("RECOMMENDATION", result["final"].upper())

    def test_state_carried_across_nodes(self):
        self.assertIn("research_brief", session.state)
        self.assertIn("draft_answer", session.state)


unittest.main(argv=["ignored", "-v"], exit=False)

test_answer_verify_refine_nodes_all_ran (__main__.TestLiveRun.test_answer_verify_refine_nodes_all_ran) ... ok
test_final_answer_is_a_recommendation (__main__.TestLiveRun.test_final_answer_is_a_recommendation) ... ok
test_state_carried_across_nodes (__main__.TestLiveRun.test_state_carried_across_nodes) ... ok
test_pass_routes_straight_to_done (__main__.TestReviewGate.test_pass_routes_straight_to_done) ... ok
test_revise_routes_to_refine_and_increments (__main__.TestReviewGate.test_revise_routes_to_refine_and_increments) ... ok
test_revision_limit_forces_done (__main__.TestReviewGate.test_revision_limit_forces_done) ... ok
test_all_four_required_agents_are_nodes (__main__.TestWorkflowWiring.test_all_four_required_agents_are_nodes) ... ok
test_answer_then_verify_pipeline (__main__.TestWorkflowWiring.test_answer_then_verify_pipeline) ... ok
test_exactly_one_terminal_node (__main__.TestWorkflowWiring.test_exactly_one_terminal_node) ... ok
test_refine_loop_closes_with_a_back_edge (__main__.T